## **SUMMARY**
This script is used to attribute local address point data into building footprints. Preprocessing of footprint, address, and parcel data is conducted in separate scripts.


In [1]:
# Copyright (c) 2025, Meredith Lochhead
# All rights reserved.
#
# This source code is licensed under the BSD 3-Clause License found in the
# LICENSE file in the root directory of this source tree.

In [2]:
# Relevant python functions
import pandas as pd
# import numpy as np
import geopandas as gpd
import os
import sys
import matplotlib.pyplot as plt
import contextily as ctx
import folium

# Import functions for inventory generation 
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
fxn_dir = os.path.join(parent_dir, "inventory_generation_functions")
sys.path.append(fxn_dir)

import functions_general as fxns
import functions_point_to_ftpt as pt_ftpt
import functions_parcel_to_ftpt as parc_ftpt ## Used in this notebook only to support initial footprint to parcel linking 

In [3]:
# Set plotting CRS values for data manipulation and plotting
crs_main = '26910' # Used for data manipulation and storage
crs_plot = '4269' # Used for plotting 

# HAYWARD BOUNDS
xbounds = [-122.15, -122.02]
ybounds = [37.60, 37.69]


In [4]:
# Target Directory 
directory = './Inventory_Outputs/Synthesized_Local/'

# Make directory and intermediate directory
os.makedirs(directory, exist_ok=True)
dir_attribution = directory + 'FootprintAttribution/'
dir_intermediate = dir_attribution + 'Intermediate/'
os.makedirs(dir_intermediate, exist_ok=True)

# Figure Directory 
fig_dir = './Figures/General/'
os.makedirs(fig_dir, exist_ok=True)

## **Load Baseline Geometry (Footprints) and Bounding Geometry (Parcels)**

In [5]:
# Load building footprints 
footprints_original = fxns.json_to_gdf('./Input_Data/ProcessedFootprints/Hayward_Footprints.json', crs_main)

# Load parcel geometry 
parcels = fxns.json_to_gdf('./Input_Data/ProcessedData/Local/Parcels.json', crs_main)

## **Tag Footprints with Possible APNs based on Geometry**


In [6]:
# Tag footprints with possible parcel APNs
# Lower bound filters rows where the overlap is less than a given % of the footprint wiht a given parcel 
# Upper bound filters rows that have at least a given % of the footprint within a single parcel (drop other parcels associated with that footprint)
print(len(footprints_original))
footprints_filtered = parc_ftpt.tag_ftpt_with_possible_apn(footprints_original, parcels, lower_bound=0, upper_bound=95)
print(len(footprints_filtered))

# Use these as footprints for merge
footprints = footprints_filtered.copy()

38300
51466


## **Attribute Points to Footprints**

In [7]:
# ## SETTINGS FOR "NOT FULL FOOTPRINT" DESIGNATION DURING MERGE PROCESS

# # Compute square footage to be used for "not full footprint" designation during merge process
#     # If most footprints have FootprintHeight available (in feet), set estimate_stories = True
#     # If most footprints do not have FootprintHeight available, using height to designate if a footprint is not full may cause bias, and it is better to just use FootprintArea (set estimate_stories = False)
# estimate_stories = False
# footprints = pt_ftpt.estimate_ftpt_size_for_merge(footprints.copy(),estimate_stories)

In [8]:
##### LOAD PREPROCESSED DATA FOR MERGE #####
points = fxns.json_to_gdf('./Input_Data/ProcessedData/Local/Address_for_Merge.json', crs_main)

##### DISPLAY NUMBER OF POINTS #####
points_length = len(points) # Used for tracking purposes 
print('NSI:', len(points))
print('Footprints:', len(footprints))

NSI: 62320
Footprints: 51466


In [9]:
##### CREATE TRACKING COLUMNS TO BE USED IN FOOTPRINT MERGE #####
points['POINT_ID'] = range(len(points)) # This is an ID number that is used throughout the script to refer to each row
points['POINT_FootprintID'] = pd.Series([pd.NA] * len(points), dtype='Int64') #None # This is the FootprintID that will be paired witht the point data throughout
points['DistanceToFtpt'] = None 
points['ClosestFtpt_ID'] = None
points['POINT_ID_List'] = points['POINT_ID'] # This tracks the ID numbers associated with that row 
points['POINT_NumPoints'] = 1 # This tracks the number of points consolidated into the single row 
points['POINT_MergeFlag'] = 0 # This tracks at what stage the point and footprint are merged


##### CREATE ADDITIONAL COLUMNS TO BE USED IN FOOTPRINT MERGE #####
points['POINT_DropFlag'] = 0 # This indicates whether a row should be dropped from the final inventory. 1 indicates yes, 0 indicates no 
points['POINT_DropNote'] = "" # Space for notes on the reason data points are dropped 
points['POINT_Source'] = 'AddressPoints' # This tracks the original data source for each row 
points['POINT_DataUpdate'] = "" # Space for notes on steps throughout update 


In [10]:
### DEFINE COLUMNS TO BE SUMMED OR COVERTED TO LISTS WHEN COMBINED ### 
# sum_columns get summed when points are merged (i.e. replacement cost)
# list_columns get put into lists (if information differs) when points are merged (i.e. foundation type)

# Set category of certain column names
excluded = ['geometry', 'CensusBlock', 'CensusTract', 'POINT_DropFlag', 'POINT_DropNote','POINT_ID',
            'POINT_FootprintID', 'DistanceToFtpt', 'ClosestFtpt_ID', 'POINT_MergeFlag','POINT_DataUpdate']
sum_columns = ['POINT_NumPoints']
list_columns = ['Address_ID', 'FeatureCode', 'FC_Hazus', 'APN_PQ','POINT_Source','POINT_ID_List']

# Print unassigned columns 
fxns.check_column_assignment(points, sum_columns, list_columns, excluded)

    

No Unassigned Columns


### **MergeFlag1 - Cases with 1 Footprint and 1 Point**

In [11]:
##### SPLIT DATA BASED ON DROP_FLAG #####
points0 = points[points['POINT_DropFlag']!=1] # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1] # Dropped - these should not be used in merge 

##### RUN FUNCTION TO MERGE CASES WITH ONE POINT WITHIN ONE FOOTPRINT #####
points0, map = pt_ftpt.merge_intersecting(points0, footprints_original, crs_plot)

# Plot overlapping footprints if found  # TODO TAKE OUT -- NOW IN PREPROCESSING 
if isinstance(map, str):
    print(map)
else: 
    display(map)

# # Update MergeFlag99 for footprints that are larger than their designated occupancy type 
# Not full footprint designation not used for local address data - thus, don't call update_mergeflag99

##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
pt_ftpt.check_post_merge_duplicates(points.copy())

##### SAVE JSON FILE #####
fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag1.json')


Points within Footprints: 55649
Unique points within Footprints (one point per footprint): 26620
Data with Associated Footprints (should match row above): 26620
Passed Check: No overlapping footprints found
Passed Check: No duplicates found
JSON File Saved


In [12]:
from IPython.display import display, HTML
def plot_help(m):
    data = m.get_root().render()
    data_fixed_height = data.replace('width: 100%;height: 100%', 'width: 100%').replace('height: 100.0%;', 'height: 609px;', 1)
    display(HTML(data_fixed_height))

In [13]:
# # ### UNCOMMENT CODE TO PLOT INTERACTIVE MAP WITH FOOTPRINTS AND POINTS
# points = fxns.json_to_gdf(dir_intermediate + '/MergeFlag1.json', crs_main)
# remaining = points[(points['POINT_DropFlag']!=1) & (points['POINT_FootprintID'].isna())]
# print(len(remaining))

# # Create a base map
# m = folium.Map(location=[footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.y, footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.x], zoom_start=12)

# # Add footprints (polygons)
# folium.GeoJson(footprints.copy().to_crs(crs_plot), color = 'gray').add_to(m)

# # Add remaining points     
# for idx, row in remaining.copy().to_crs(crs_plot).iterrows():
#     folium.CircleMarker(location=[row.geometry.y, row.geometry.x], 
#                         radius=1, 
#                         color='blue', 
#                         fill=True, 
#                         fill_color='blue').add_to(m)

# # display(m)

### **MergeFlag2 - Address Cases with 1 Footprint and Multiple Points**

In [14]:
##### LOAD DATA #####
points = fxns.json_to_gdf(dir_intermediate + 'MergeFlag1.json', crs_main)


##################################################################################################################################

############# THE FOLLOWING CAN BE FILLED OUT TO MAKE MODIFICATIONS BASED ON THE PRINTED OUT "ODD OCCUPANCY PAIRINGS" #################

##### MANUALLY ASSIGN OCCUPANCY CLASS FOR FOOTPRINTS HERE #####
manually_assigned_occupancy_data = {
    "FootprintID":[], # Put FootprintID values in this list
    "POINT_OccupancyClass":[]} # Put corresponding occupancy class values in this list. Can enter string or list, 
                             # i.e. "NSI_OccupancyClass":['RES1',['IND3','RES3C]] would correspond to first and second FootprintID entered in list above
manually_assigned_occupancy = pd.DataFrame(manually_assigned_occupancy_data)

##### MANUALLY DROP NSI POINTS HERE #####
ids_to_drop = [] # Place NSI_ID values here that should be dropped from the merge 
points = pt_ftpt.drop_ids(points, ids_to_drop, 'Manually dropped due to occupancy class incompatibility')

##################################################################################################################################


##### SPLIT DATA BASED ON DROP_FLAG #####
points0 = points[points['POINT_DropFlag']!=1] # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1] # Dropped - these should not be used in merge 


#### RUN FUNCTION TO MERGE CASES WITH MULTIPLE POINTS WITHIN ONE FOOTPRINT #####
# Set flag to print odd occupancy pairings, including (RES + IND), (RES + GOV), and (EDU + IND) - does not change function outputs, only displays 
print_odd_occupancy_pairings = False
use_size_limit = False
use_nsi_occupancy_merge = False
points0 = pt_ftpt.address_overlapping_points(points0.copy(), footprints_original.copy(), list_columns, sum_columns, manually_assigned_occupancy,use_size_limit, use_nsi_occupancy_merge, print_odd_occupancy_pairings, crs_plot)

##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
points['POINT_FootprintID'] = points['POINT_FootprintID'].astype('Int64')
pt_ftpt.check_post_merge_duplicates(points.copy())

##### SAVE JSON FILE #####
fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag2.json')

Number of Points Remaining: 35700
Number of Footprints Remaining: 11680
Number of Points within Footprint Polygons: 29050
Number of Footprints with Multiple Points (Looping Through These Now): 4294
10% complete
20% complete
30% complete
40% complete
50% complete
60% complete
70% complete
80% complete
90% complete
100% complete
Passed Check: No duplicates found
JSON File Saved


In [15]:
# # ### UNCOMMENT CODE TO PLOT INTERACTIVE MAP WITH FOOTPRINTS AND POINTS
# points = pt_ftpt.json_to_gdf(dir_intermediate + '/MergeFlag2.json', crs_main)
# remaining = points[(points['POINT_DropFlag']!=1) & (points['POINT_FootprintID'].isna())]
# print(len(remaining))
# fig, ax = plt.subplots(figsize=(30,30))
# footprints.plot(ax=ax, color = 'gray', alpha = 0.3)
# remaining.plot(ax=ax, markersize = 1)
# plt.show()

In [16]:
# # ### UNCOMMENT CODE TO PLOT INTERACTIVE MAP WITH FOOTPRINTS AND POINTS
# points = pt_ftpt.json_to_gdf(dir_intermediate + '/MergeFlag2.json', crs_main)
# remaining = points[(points['POINT_DropFlag']!=1) & (points['POINT_FootprintID'].isna())]
# print(len(remaining))

# # Create a base map
# m = folium.Map(location=[footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.y, footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.x], zoom_start=12)

# # Add footprints (polygons)
# folium.GeoJson(footprints.copy().to_crs(crs_plot), color = 'gray').add_to(m)

# # Add remaining points     
# for idx, row in remaining.copy().to_crs(crs_plot).iterrows():
#     folium.CircleMarker(location=[row.geometry.y, row.geometry.x], 
#                         radius=1, 
#                         color='blue', 
#                         fill=True, 
#                         fill_color='blue').add_to(m)

# display(m)

### **MergeFlag350 - Merge Points within 50m of a Footprint**

In [17]:
# Load data 
points = fxns.json_to_gdf(dir_intermediate + '/MergeFlag2.json', crs_main)


##################################################################################################################################

############# THE FOLLOWING CAN BE FILLED OUT TO MAKE MODIFICATIONS BASED ON THE PRINTED OUT "ODD OCCUPANCY PAIRINGS" #################

##### MANUALLY ASSIGN OCCUPANCY CLASS FOR FOOTPRINTS HERE #####
manually_assigned_occupancy_data = {
    "FootprintID":[], # Put FootprintID values in this list
    "POINT_OccupancyClass":[]} # Put corresponding occupancy class values in this list. Can enter string or list, 
                             # i.e. "NSI_OccupancyClass":['RES1',['IND3','RES3C]] would correspond to first and second FootprintID entered in list above
manually_assigned_occupancy = pd.DataFrame(manually_assigned_occupancy_data)

##### MANUALLY DROP NSI POINTS HERE #####
ids_to_drop = [] # Place NSI_ID values here that should be dropped from the merge 
points = pt_ftpt.drop_ids(points, ids_to_drop, 'Manually dropped due to occupancy class incompatibility')

##################################################################################################################################


##### SPLIT DATA BASED ON DROP_FLAG #####
points0 = points[points['POINT_DropFlag']!=1].copy() # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1].copy() # Dropped - these should not be used in merge 


##### CONDUCT MERGE #####

# Get list of bounding geometries 
bounding_id_list =  parcels['APN_PQ'].unique()
bounding_id_name = 'APN_PQ'
bounding_geometry = parcels.copy()

# Merge data using function 

points0 = pt_ftpt.distance_limit_merge(bounding_id_list, points0.copy(), footprints, bounding_id_name, manually_assigned_occupancy, list_columns, sum_columns, bounding_geometry, crs_plot,
                            distance_limit = 50, # Meters
                            use_surrounding_bgs = False, # Footprints in surrounding bounding geometries (parcels) will be considered for each NSI point
                            prioritize_empty_footprints = False, # Empty footprints within the distance limit will be prioritized over partially full or full footprints for attribution 
                            prioritize_partial_footprints = False, # Footprints with MergeFlag = 99 will be prioritized over full footprints for attribution (prioritized second to empty footprints if prioritize_empty_footprints = True)
                            use_full_footprints = True, # Full footprints will be considered for atribution (once empty and partial footprints in distance limit have been exhausted, if the above prioritize flags are set to True)
                            merge_flag = 350, 
                            use_size_limit = False, # Use size limit to designate "partially full footprints"
                            use_nsi_occupancy_merge = False, # Set to True only if merging NSI data 
                            print_odd_occupancy_pairings = False) # If True, some occupancy class pairs will be printed out if they are merged into same footprint to manually check 


##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
points['POINT_FootprintID'] = points['POINT_FootprintID'].astype('Int64')
pt_ftpt.check_post_merge_duplicates(points.copy())

##### SAVE JSON FILE #####
fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag350.json')


Number of Points Remaining: 6650
Number of Footprints Remaining: 9069
Processing 40080 Bounding Geometries
10% complete
20% complete
30% complete
40% complete
50% complete
60% complete
70% complete
80% complete
90% complete
100% complete
Passed Check: No duplicates found
JSON File Saved


In [18]:
# # ### UNCOMMENT CODE TO PLOT INTERACTIVE MAP WITH FOOTPRINTS AND POINTS
# points = pt_ftpt.json_to_gdf(dir_intermediate + '/MergeFlag350.json', crs_main)
# remaining = points[(points['POINT_DropFlag']!=1) & (points['POINT_FootprintID'].isna())]
# print(len(remaining))
# fig, ax = plt.subplots(figsize=(30,30))
# footprints.plot(ax=ax, color = 'gray', alpha = 0.3)
# remaining.plot(ax=ax, markersize = 1)
# plt.show()

In [19]:
# # ### UNCOMMENT CODE TO PLOT INTERACTIVE MAP WITH FOOTPRINTS AND POINTS
# points = pt_ftpt.json_to_gdf(dir_intermediate + '/MergeFlag350.json', crs_main)
# remaining = points[(points['POINT_DropFlag']!=1) & (points['POINT_FootprintID'].isna())]å
# print(len(remaining))

# # Create a base map
# m = folium.Map(location=[footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.y, footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.x], zoom_start=12)

# # Add footprints (polygons)
# folium.GeoJson(footprints.copy().to_crs(crs_plot), color = 'gray').add_to(m)
# folium.GeoJson(parcels.copy().to_crs(crs_plot), color = 'blue').add_to(m)

    
#     # Add remaining points     
# for idx, row in remaining.copy().to_crs(crs_plot).iterrows():
#     folium.CircleMarker(location=[row.geometry.y, row.geometry.x], 
#                         radius=1, 
#                         color='red', 
#                         fill=True, 
#                         fill_color='red',
#                         popup = row['FeatureCode']).add_to(m)

# # display(m)

## **Drop Relevant Points and Export Data**

In [20]:
## Load data
points = fxns.json_to_gdf(dir_intermediate + '/MergeFlag350.json', crs_main)
print(len(points))

## Drop points that are marked to be dropped 
points = points[points['POINT_DropFlag']!=1]
print(len(points))

# Find APNs that contain footprints 
parcels_with_footprints = footprints['APN_PQ'].unique()

# Drop address points that are in parcels that contain footprints, but were not attributed to a footprint
points = points[~(points['APN_PQ'].isin(parcels_with_footprints) & points['POINT_FootprintID'].isna())]

62320
34894


In [21]:
# Drop additional columns used for tracking purposes 
points = points.drop(columns = ['POINT_DropFlag','POINT_DropNote','DistanceToFtpt', 'ClosestFtpt_ID','POINT_ID', 'POINT_DataUpdate'])
points = points.rename(columns={'POINT_ID_List': 'POINT_ID'})

# Rename footprints 
points = points.rename(columns={'POINT_FootprintID': 'FootprintID'})

##### SAVE JSON FILE #####
fxns.gdf_to_json(points.copy(), dir_attribution + 'Address_Data_Attributed.json')

JSON File Saved
